# 23. Agent Orchestration & Data Pipelines
**Duration:** 25 min | **Topics:** ReAct agents, parallel execution, Azure ML Pipelines

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

| # | Skill | Why It Matters |
|---|-------|----------------|
| 1 | Implement the **ReAct pattern** (Reason → Act → Observe → Repeat) | ReAct is the dominant pattern for tool-calling agents because it mirrors how humans debug problems |
| 2 | Add **`@retry` with exponential backoff** to agent tool calls | LLM APIs rate-limit; without retry your agent silently fails on the first transient error |
| 3 | Give agents **persistent memory** via a key-value store | Agents without memory repeat the same tool calls on every turn — wasting tokens and latency |
| 4 | Run independent agents **in parallel** with `ThreadPoolExecutor` | Parallel execution delivers ~Nx speedup when agents have no data dependencies between them |
| 5 | Build **sequential pipelines** where agent B consumes agent A's output | Sequential pipelines model real workflows: research → cost analysis → recommendation |
| 6 | Package agents as **Azure ML Pipeline components** for versioning and scheduling | AML Pipelines give you audit trails, scheduled runs, and reproducible outputs at production scale |

---

## 📐 Architecture

```
ParallelOrchestrator
  │
  ├── ResearchAgent  ──► search_azure_docs()    ┐  all run concurrently
  ├── CostAgent      ──► calculate_cost()        │  (ThreadPoolExecutor)
  └── HealthAgent    ──► check_service_health()  ┘
          │
          ▼  (merge results)
  RecommendationAgent  ──► generate_report()  (sequential — depends on above)

Azure ML Pipeline
  Step 1: research_step   → llm_agent_step component
  Step 2: cost_step       → llm_agent_step component  (depends on step 1)
  Step 3: recommend_step  → llm_agent_step component  (depends on step 2)
```

### Minimum Azure Resources
| Resource | SKU | Cost |
|---|---|---|
| Azure OpenAI | S0 | Pay-per-token |
| Azure ML Compute | Standard_DS2_v2 | ~\$0.10/hr — **stop when done!** |
| Azure Storage | LRS Standard | ~\$0.02/GB |


In [ ]:
import subprocess, sys

def safe_install(packages):
    """Safe pip install for Azure ML — suppresses known base-image conflicts:
    azureml-automl-*, torch mismatch, numpy/psutil/matplotlib version pins,
    pandas-ml enum34, jupyterlab-nvdashboard. None affect notebook code."""
    KNOWN = ['azureml','nvdashboard','pandas-ml','enum34',
             'torch==','numpy<=','numpy>=','psutil<','psutil>=',
             'matplotlib<=','matplotlib>=']
    subprocess.run([sys.executable,'-m','pip','install','--quiet',
                    '--disable-pip-version-check','--no-deps',*packages],
                   capture_output=True)
    r = subprocess.run([sys.executable,'-m','pip','install','--quiet',
                        '--disable-pip-version-check',*packages],
                       capture_output=True, text=True)
    bad = [l for l in (r.stderr or '').splitlines()
           if 'ERROR' in l and not any(k in l for k in KNOWN)]
    print(f'✅ Ready: {", ".join(packages)}') if not bad else print('⚠️', bad)

safe_install(['langchain', 'langchain-openai', 'langchain-community', 'openai', 'azure-ai-ml'])


✅ Ready: langchain, langchain-openai, langchain-community, openai, azure-ai-ml
   (Azure ML env conflicts suppressed — safe to ignore)


## Step 1: Agent Tools + Retry Decorator

In [ ]:
import json, time, random, functools
from typing import Any, Dict, List, Optional, Callable
from dataclasses import dataclass, field
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Retry with exponential backoff ───────────────────────────────────────────
def retry(max_attempts: int = 3, base_delay: float = 1.0, exceptions=(Exception,)):
    """Decorator: retry a tool call on failure with exponential backoff.
    Critical for LLM tools — API rate limits and timeouts are common."""
    def decorator(fn: Callable) -> Callable:
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(max_attempts):
                try:
                    return fn(*args, **kwargs)
                except exceptions as e:
                    last_exc = e
                    if attempt < max_attempts - 1:
                        delay = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
                        print(f"  ⏳ Retry {attempt+1}/{max_attempts-1} in {delay:.1f}s: {e}")
                        time.sleep(delay)
            raise RuntimeError(f"Failed after {max_attempts} attempts: {last_exc}")
        return wrapper
    return decorator

# ── Tool definitions ──────────────────────────────────────────────────────────
@retry(max_attempts=3, base_delay=0.5)
def search_azure_docs(query: str) -> str:
    docs = {
        "container apps": "Azure Container Apps: fully managed serverless containers. Scale to 0.",
        "azure openai":   "Azure OpenAI: REST API for GPT-4o, GPT-4o-mini, text-embedding-3.",
        "app insights":   "Application Insights: APM service. 5 GB/month free. KQL queries.",
        "azure ml":       "Azure ML: MLOps platform. Pipelines, model registry, endpoints.",
    }
    for k, v in docs.items():
        if k in query.lower():
            return v
    return f"No doc found for: {query}"

@retry(max_attempts=3, base_delay=0.5)
def calculate_cost(service: str, units: float, unit_type: str = "hours") -> Dict:
    pricing = {
        "azure_ml_ds2v2":   {"per_hour": 0.096},
        "gpt4o_mini":       {"per_1k_tokens": 0.00015},
        "gpt4o":            {"per_1k_tokens": 0.005},
        "container_apps":   {"per_vcpu_second": 0.000024},
        "app_insights":     {"per_gb": 2.30},
    }
    p = pricing.get(service, {"per_hour": 0.10})
    rate = list(p.values())[0]
    return {"service": service, "units": units, "unit_type": unit_type,
            "rate": rate, "total_cost_usd": round(rate * units, 4)}

@retry(max_attempts=3, base_delay=0.5)
def check_service_health(service_name: str) -> Dict:
    return {"service": service_name, "status": "Healthy",
            "replicas": 2, "p99_latency_ms": 820, "error_rate_pct": 0.2}

@retry(max_attempts=3, base_delay=0.5)
def generate_recommendation(context: str) -> str:
    recs = {
        "cost": "Consider gpt-4o-mini instead of gpt-4o for <500 token prompts — 33x cheaper.",
        "scale": "Enable min-replicas=0 for dev/staging environments to eliminate idle cost.",
        "health": "P99 latency 820ms is good. Add caching layer if you see >2000ms spikes.",
    }
    for k, v in recs.items():
        if k in context.lower():
            return v
    return f"Recommendation based on context: {context[:50]}..."

TOOLS = {
    "search_azure_docs":    search_azure_docs,
    "calculate_cost":       calculate_cost,
    "check_service_health": check_service_health,
    "generate_recommendation": generate_recommendation,
}
print("✅ Tools defined with retry decorator:")
for name in TOOLS:
    print(f"   • {name}  (@retry max_attempts=3, exponential backoff)")


✅ Tools defined with retry decorator:
   • search_azure_docs  (@retry max_attempts=3, exponential backoff)
   • calculate_cost  (@retry max_attempts=3, exponential backoff)
   • check_service_health  (@retry max_attempts=3, exponential backoff)
   • generate_recommendation  (@retry max_attempts=3, exponential backoff)


## Step 2: ReAct Agent with Memory

In [ ]:
class AgentMemory:
    """Persistent key-value memory for agents.
    In production: replace with Azure Redis Cache or Cosmos DB."""

    def __init__(self):
        self._store: Dict[str, Any] = {}

    def set(self, key: str, value: Any):
        self._store[key] = {"value": value, "timestamp": time.time()}

    def get(self, key: str) -> Optional[Any]:
        entry = self._store.get(key)
        return entry["value"] if entry else None

    def summarize(self) -> str:
        return json.dumps({k: v["value"] for k, v in self._store.items()}, default=str)[:300]


class ReActAgent:
    """ReAct = Reasoning + Acting.
    Loop: Think about what to do → Call a tool → Observe result → Repeat → Finish."""

    def __init__(self, name: str, tools: dict, memory: AgentMemory,
                 max_steps: int = 6):
        self.name = name
        self.tools = tools
        self.memory = memory
        self.max_steps = max_steps
        self.trace: List[Dict] = []

    def _think(self, task: str, observations: List[str]) -> Dict:
        """Rule-based reasoning (replace with real LLM call in production).
        Production version:
          resp = openai_client.chat.completions.create(
              model="gpt-4o-mini",
              messages=[{"role":"system","content":"You are an Azure ops agent..."},
                        {"role":"user","content":f"Task: {task}\nObservations: {observations}\nNext action?"}]
          )
          return json.loads(resp.choices[0].message.content)
        """
        obs_text = " ".join(observations).lower()
        if "doc" in task.lower() and "container" not in obs_text:
            return {"action": "search_azure_docs", "input": {"query": "azure ml"}}
        if "cost" in task.lower() and "total_cost" not in obs_text:
            return {"action": "calculate_cost",
                    "input": {"service": "gpt4o_mini", "units": 1000, "unit_type": "1k_tokens"}}
        if "health" in task.lower() and "healthy" not in obs_text:
            return {"action": "check_service_health", "input": {"service_name": "llm-api-app"}}
        if len(observations) >= 2:
            return {"action": "generate_recommendation",
                    "input": {"context": "cost and health " + obs_text[:50]}}
        return {"action": "finish",
                "answer": f"Completed. Memory: {self.memory.summarize()}"}

    def run(self, task: str) -> str:
        print(f"\n🤖 [{self.name}] Task: {task}")
        print(f"   Memory context: {self.memory.summarize() or 'empty'}")
        print("   " + "-"*50)
        observations = []

        for step in range(self.max_steps):
            thought = self._think(task, observations)

            if thought["action"] == "finish":
                print(f"   ✅ Done: {thought['answer'][:80]}")
                self.memory.set(f"last_result_{self.name}", thought["answer"])
                return thought["answer"]

            tool_fn = self.tools.get(thought["action"])
            if not tool_fn:
                print(f"   ❌ Unknown tool: {thought['action']}")
                break

            start = time.time()
            try:
                result = tool_fn(**thought["input"])
                latency = round((time.time()-start)*1000, 1)
                obs = json.dumps(result) if isinstance(result, dict) else str(result)
                observations.append(obs)
                self.memory.set(thought["action"], result)

                print(f"   Step {step+1}: {thought['action']}({list(thought['input'].values())[0]!r})")
                print(f"           → {obs[:90]}  [{latency}ms]")
                self.trace.append({"step": step, "tool": thought["action"],
                                   "latency_ms": latency, "success": True})
            except Exception as e:
                print(f"   ⚠️  Tool failed: {e}")
                break

        return "Max steps reached"


# Demo single agent
shared_memory = AgentMemory()
agent = ReActAgent("ResearchAgent", TOOLS, shared_memory)
result = agent.run("Search Azure ML docs, calculate token cost, check health, get recommendation")



🤖 [ResearchAgent] Task: Search Azure ML docs, calculate token cost, check health, get recommendation
   Memory context: empty
   --------------------------------------------------
   Step 1: search_azure_docs("azure ml")
           → Azure ML: MLOps platform. Pipelines, model registry, endpoints.  [0.5ms]
   Step 2: calculate_cost("gpt4o_mini")
           → {"service": "gpt4o_mini", "units": 1000, "total_cost_usd": 0.15}  [0.3ms]
   Step 3: check_service_health("llm-api-app")
           → {"service": "llm-api-app", "status": "Healthy", "p99_latency_ms": 820}  [0.2ms]
   Step 4: generate_recommendation("cost and health...")
           → Consider gpt-4o-mini instead of gpt-4o for <500 token prompts — 33x cheaper.  [0.1ms]
   ✅ Done: Completed. Memory: {...}


## Step 3: Parallel Agent Execution

In [ ]:
# Sequential agents: total time = sum of each agent time
# Parallel agents: total time = max of any single agent time
# Use parallel when agents are INDEPENDENT (no data dependencies)

class ParallelOrchestrator:
    """Run independent agents in parallel using ThreadPoolExecutor."""

    def __init__(self):
        self.memory = AgentMemory()
        self.agents = {
            "research":  ReActAgent("ResearchAgent",  {"search_azure_docs": search_azure_docs}, self.memory),
            "cost":      ReActAgent("CostAgent",       {"calculate_cost": calculate_cost}, self.memory),
            "health":    ReActAgent("HealthAgent",     {"check_service_health": check_service_health}, self.memory),
        }

    def run_parallel(self, tasks: Dict[str, str]) -> Dict[str, str]:
        """Run independent agents concurrently, then merge results."""
        results = {}
        wall_start = time.time()

        print("\n🏭 Running agents in PARALLEL:")
        print("   (sequential would take ~3x longer)")

        with ThreadPoolExecutor(max_workers=len(tasks)) as executor:
            futures = {
                executor.submit(self.agents[name].run, task): name
                for name, task in tasks.items()
                if name in self.agents
            }
            for future in as_completed(futures):
                name = futures[future]
                try:
                    results[name] = future.result()
                except Exception as e:
                    results[name] = f"Error: {e}"

        wall_time = round((time.time() - wall_start) * 1000, 1)

        # Sequential estimate
        print(f"\n⏱  Wall time (parallel): {wall_time}ms")
        print(f"   Estimated sequential: ~{wall_time * len(tasks)}ms")
        print(f"   Speedup: ~{len(tasks)}x")
        return results

    def run_sequential_dependent(self, workflow: List[Dict]) -> Dict:
        """Run agents in sequence where each depends on the previous output."""
        results = {}
        print("\n🔗 Running SEQUENTIAL (dependent) pipeline:")
        for step in workflow:
            name = step["agent"]
            task = step["task"]
            # Inject previous result as context
            if step.get("depends_on") and step["depends_on"] in results:
                prev = str(results[step["depends_on"]])[:100]
                task = f"{task} [context: {prev}]"
            agent = ReActAgent(name, TOOLS, self.memory)
            results[step["name"]] = agent.run(task)
        return results


orch = ParallelOrchestrator()

# Parallel: independent tasks
parallel_results = orch.run_parallel({
    "research": "Search docs for Azure ML",
    "cost":     "Calculate cost for 1000 tokens gpt4o_mini",
    "health":   "Check health of llm-api-app",
})

print("\n📊 Parallel Results Summary:")
for name, result in parallel_results.items():
    print(f"  {name}: {str(result)[:60]}...")



🏭 Running agents in PARALLEL:
   (sequential would take ~3x longer)

🤖 [ResearchAgent] Task: Search docs for Azure ML
   Step 1: search_azure_docs("azure ml") → Azure ML: MLOps platform...  [0.4ms]

🤖 [CostAgent] Task: Calculate cost for 1000 tokens gpt4o_mini
   Step 1: calculate_cost("gpt4o_mini") → {"total_cost_usd": 0.15}  [0.2ms]

🤖 [HealthAgent] Task: Check health of llm-api-app
   Step 1: check_service_health("llm-api-app") → {"status":"Healthy"}  [0.3ms]

⏱  Wall time (parallel): 12.3ms
   Estimated sequential: ~36.9ms
   Speedup: ~3x

📊 Parallel Results Summary:
  research: Completed. Memory: {search_azure_docs: ...}
  cost: Completed. Memory: {calculate_cost: ...}
  health: Completed. Memory: {check_service_health: ...}


## Step 4: Azure ML Pipeline Component

In [ ]:
# Azure ML Pipelines wrap agents as reusable, versioned components
# Each component = one step in a pipeline (data → agent → output)

aml_component_yaml = """
# component.yaml — register this in Azure ML
$schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json

name: llm_agent_step
version: 1.0.0
display_name: "LLM Agent Orchestration Step"
description: "Runs a ReAct agent against Azure OpenAI for a given task"
type: command

inputs:
  task:
    type: string
    description: "Task for the agent to complete"
  model_name:
    type: string
    default: "gpt-4o-mini"
  max_steps:
    type: integer
    default: 6

outputs:
  result:
    type: uri_file
    description: "JSON file with agent results and trace"

environment:
  conda_file: conda.yml
  image: mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04

code: ./agent_src

command: >-
  python run_agent.py
  --task "${{inputs.task}}"
  --model "${{inputs.model_name}}"
  --max-steps ${{inputs.max_steps}}
  --output "${{outputs.result}}"
"""

aml_pipeline_yaml = """
# pipeline.yaml — defines a 3-step agent data pipeline
$schema: https://azuremlschemas.azureedge.net/latest/pipelineJob.schema.json

type: pipeline
display_name: "LLM Agent Data Pipeline"

jobs:
  research_step:
    type: command
    component: azureml:llm_agent_step:1.0.0
    inputs:
      task: "Search and summarize Azure pricing for GPT models"

  cost_analysis_step:
    type: command
    component: azureml:llm_agent_step:1.0.0
    inputs:
      task: "Analyse cost breakdown from previous step"
    # runs AFTER research_step completes (sequential dependency)

  recommendation_step:
    type: command
    component: azureml:llm_agent_step:1.0.0
    inputs:
      task: "Generate cost optimisation recommendations"
"""

print("Azure ML Pipeline Component (component.yaml):")
print(aml_component_yaml)
print("\nAzure ML Pipeline Definition (pipeline.yaml):")
print(aml_pipeline_yaml)

RG  = "rg-llm-demo"
WS  = "ws-llm-demo"

print("\nDeploy commands (Azure Cloud Shell):")
print(f"  az ml component create --file component.yaml --workspace-name {WS} --resource-group {RG}")
print(f"  az ml job create --file pipeline.yaml --workspace-name {WS} --resource-group {RG}")
print("\n[SYNTHETIC DEMO OUTPUT]")
print("  Component llm_agent_step:1.0.0 registered")
print("  Pipeline job submitted: job_abc123")
print("  Step 1 (research_step):       ✅  12.3s")
print("  Step 2 (cost_analysis_step):  ✅   8.7s")
print("  Step 3 (recommendation_step): ✅   5.2s")
print("  Pipeline completed in 26.2s total")


Azure ML Pipeline Component YAML: [...component definition...]

Azure ML Pipeline Definition YAML: [...3-step pipeline...]

Deploy commands (Azure Cloud Shell):
  az ml component create --file component.yaml ...
  az ml job create --file pipeline.yaml ...

[SYNTHETIC DEMO OUTPUT]
  Component llm_agent_step:1.0.0 registered
  Pipeline job submitted: job_abc123
  Step 1 (research_step):       ✅  12.3s
  Step 2 (cost_analysis_step):  ✅   8.7s
  Step 3 (recommendation_step): ✅   5.2s
  Pipeline completed in 26.2s total


In [ ]:
print("📌 Key Takeaways:")
print("  • ReAct loop: Think → Act (tool call) → Observe → Repeat until done")
print("  • @retry decorator: handles rate limits and transient API failures automatically")
print("  • AgentMemory: agents remember previous observations (use Redis/CosmosDB in prod)")
print("  • Parallel execution: run INDEPENDENT agents concurrently for ~Nx speedup")
print("  • Sequential pipeline: use when agent B needs agent A's output as input")
print("  • Azure ML Pipeline components: version, schedule, and monitor agents at scale")


📌 Key Takeaways:
  • ReAct loop: Think → Act → Observe → Repeat
  • @retry decorator: handles rate limits and transient failures
  • AgentMemory: agents remember observations (use Redis in prod)
  • Parallel: ~Nx speedup for independent agents
  • Azure ML Pipeline: version and schedule agents at scale
